# MuseTalk — Audio-Driven Lip Sync
**InterVyu.ai | Avatar-Video**

Based on [TMElyralab/MuseTalk](https://github.com/TMElyralab/MuseTalk).

**Environments supported:**
- Google Colab (Tesla T4/V100, Python 3.12)
- Local macOS/Linux with  (Python 3.11)


## 0. Repo Setup (Colab only)
Skip this cell if running locally.

In [14]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Option A: clone from GitHub (replace with your repo URL)
    !git clone https://github.com/InterVyuMitr-ai/Avatar-Video.git
    %cd Avatar-Video

    # Option B: mount Google Drive
    # from google.colab import drive
    # drive.mount("/content/drive")
    # %cd /content/drive/MyDrive/InterVyu.ai/Avatar-Video

print(f"Working directory: {os.getcwd()}")
print(f"Running in Colab: {IN_COLAB}")

Cloning into 'Avatar-Video'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 25 (delta 4), reused 25 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 132.29 KiB | 22.05 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Avatar-Video/Avatar-Video
Working directory: /content/Avatar-Video/Avatar-Video
Running in Colab: True


## 1. Install Dependencies

In [15]:
import sys, subprocess, torch
IN_COLAB = "google.colab" in sys.modules

def try_install_mmcv(pip="pip"):
    """Best-effort mmcv install. Not required — pipeline falls back to Haar cascade."""
    cuda = "cu" + torch.version.cuda.replace(".", "")[:3] if torch.cuda.is_available() else "cpu"
    tv   = "torch" + ".".join(torch.__version__.split("+")[0].split(".")[:2])
    url  = f"https://download.openmmlab.com/mmcv/dist/{cuda}/{tv}/index.html"
    candidates = [
        [pip, "install", "-q", "mmcv==2.0.1", "-f", url],
        [pip, "install", "-q", "mmcv==2.1.0", "-f", url],
        [pip, "install", "-q", "mmcv"],
    ]
    for cmd in candidates:
        r = subprocess.run(cmd, capture_output=True)
        if r.returncode == 0:
            print(f"mmcv installed via: {" ".join(cmd[-2:])}")
            return
    print("WARNING: mmcv could not be installed. DWPose unavailable; using Haar cascade fallback.")

if IN_COLAB:
    # diffusers/transformers unpinned — Colab ships huggingface-hub>=1.0
    # which breaks diffusers<0.31 and transformers<4.40
    !pip install -q "diffusers>=0.31.0" "transformers>=4.40.0" accelerate==0.28.0 "huggingface_hub>=1.0" opencv-python soundfile librosa einops gradio omegaconf ffmpeg-python moviepy "imageio[ffmpeg]" gdown
    !pip install -q mmengine
    !pip install -q mmdet mmpose
    try_install_mmcv("pip")
else:
    !uv sync
    !uv run pip install -q mmengine mmdet mmpose
    try_install_mmcv("uv")

print("Install complete.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.39.2 which is incompatible.
         Pipeline will use Haar cascade fallback — results may be less accurate.
Install complete.


## 2. Download Model Weights (~4 GB)

In [16]:
!bash download_models.sh

Fetching 2 files: 100% 2/2 [00:25<00:00, 12.99s/it]
Download complete: 100% 3.40G/3.40G [00:25<00:00, 131MB/s]                
Fetching 2 files: 100% 2/2 [00:36<00:00, 18.17s/it]
Download complete: 100% 3.40G/3.40G [00:36<00:00, 93.5MB/s]               
Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 2 files: 100% 2/2 [00:04<00:00,  2.25s/it]
Download complete: 100% 335M/335M [00:04<00:00, 74.1MB/s]               
Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 3 files: 100% 3/3 [00:03<00:00,  1.17s/it]
Download complete: : 151MB [00:03, 42.8MB/s]              
Fetching 1 files: 100% 1/1 [00:04<00:00,  4.30s/it]
Download complete: 100% 407M/407M [00:04<00:00, 94.2MB/s]               
Fetching 1 files:   0% 

## 3. Verify GPU

In [17]:
import torch
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif torch.backends.mps.is_available():
    print("Apple MPS (Metal) — will work, slower than CUDA")
else:
    print("CPU only — very slow, GPU strongly recommended")

CUDA: Tesla T4
VRAM: 15.6 GB


## 4. Load Pipeline

In [18]:
from musetalk_pipeline import MuseTalkPipeline, MuseTalkConfig

VERSION = "v1"  # "v1" or "v15" (v15 = higher quality, needs more VRAM)

cfg = MuseTalkConfig(version=VERSION)
if VERSION == "v15":
    cfg.unet_model_path  = "models/musetalkV15/unet.pth"
    cfg.unet_config_path = "models/musetalkV15/musetalk.json"

pipe = MuseTalkPipeline(cfg)
pipe.load_models()
print("Pipeline ready!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

RuntimeError: Failed to import diffusers.models.autoencoders.autoencoder_kl because of the following error (look up to see its traceback):
huggingface-hub>=0.19.3,<1.0 is required for a normal functioning of this module, but found huggingface-hub==1.7.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

## 5. Run Inference

In [ ]:
VIDEO_PATH  = "data/video/source.mp4"
AUDIO_PATH  = "data/audio/target.wav"
OUTPUT_PATH = "results/output.mp4"
BBOX_SHIFT  = 0  # tune if lips look misaligned (-10 to 10)

result = pipe.run(
    video_path=VIDEO_PATH,
    audio_path=AUDIO_PATH,
    output_path=OUTPUT_PATH,
    bbox_shift=BBOX_SHIFT,
)
print(f"Saved to: {result}")

## 6. Preview

In [ ]:
from IPython.display import Video
Video(OUTPUT_PATH, embed=True, width=640)

## 7. Batch Inference via YAML Config

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
run_prefix = "" if IN_COLAB else "uv run "
!{run_prefix}python -m scripts.inference \n
    --inference_config configs/inference/test.yaml \n
    --result_dir results/batch \n
    --version v1

## 8. Gradio Web UI

In Colab, enable  for a public tunnel link.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Patch app.py to use share=True for Colab tunnel
    !sed -i "s/demo.launch(share=False)/demo.launch(share=True)/" app.py
run_prefix = "" if IN_COLAB else "uv run "
!{run_prefix}python app.py

## Tips

| Issue | Fix |
|---|---|
| Lips misaligned | Adjust  (-10 to +10) |
| Low resolution | Post-process with GFPGAN super-resolution |
| Flickering | Use  |
| GPU OOM | Reduce  in  |
| Colab disconnects | Save models to Google Drive and reload |
| Slow on MPS | Apple M-series: expect ~6–10 fps vs 30 fps on CUDA |
